# Propensity Score Matching - L'Oreal India Example

This notebook demonstrates propensity score matching to estimate the causal effect of beauty advisors on store sales.

**Problem**: Beauty advisors are placed in high-potential stores, creating selection bias.

**Solution**: Use propensity score matching to create comparable groups.

**True Effect**: Rs 5 Lakhs/month (we'll see if we can recover this)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Generate Store Data with Selection Bias

In reality, beauty advisors are NOT randomly assigned. They go to:
- High footfall stores
- Larger stores
- Mall locations
- Tier 1 cities

This creates **selection bias** when comparing sales.

In [ ]:
def generate_store_data(n_stores=500, seed=42):
    """Generate synthetic store data with selection bias."""
    np.random.seed(seed)
    
    # Store characteristics
    footfall = np.random.exponential(300, n_stores) + 100
    store_size = np.random.normal(800, 200, n_stores).clip(300, 1500)
    is_mall = np.random.binomial(1, 0.3, n_stores)
    city_tier = np.random.choice([1, 2, 3], n_stores, p=[0.3, 0.4, 0.3])
    
    # Beauty advisor assignment (BIASED toward better stores)
    advisor_propensity = (
        0.002 * footfall
        + 0.001 * store_size
        + 0.3 * is_mall
        - 0.15 * city_tier
        + np.random.normal(0, 0.3, n_stores)
    )
    advisor_prob = 1 / (1 + np.exp(-advisor_propensity + 1))
    has_advisor = np.random.binomial(1, advisor_prob)
    
    # Sales (affected by characteristics AND advisor)
    TRUE_ADVISOR_EFFECT = 5  # Rs 5 Lakhs - this is what we want to estimate
    sales = (
        10  # Base
        + 0.05 * footfall
        + 0.02 * store_size
        + 8 * is_mall
        - 3 * city_tier
        + TRUE_ADVISOR_EFFECT * has_advisor  # CAUSAL EFFECT
        + np.random.normal(0, 3, n_stores)
    ).clip(5, None)
    
    return pd.DataFrame({
        'store_id': range(n_stores),
        'footfall': footfall,
        'store_size': store_size,
        'is_mall': is_mall,
        'city_tier': city_tier,
        'has_advisor': has_advisor,
        'sales_lakhs': sales
    })

df = generate_store_data()

print("Store Data Summary")
print("=" * 50)
print(f"Total stores: {len(df)}")
print(f"Stores with advisor: {df['has_advisor'].sum()}")
print(f"Stores without advisor: {len(df) - df['has_advisor'].sum()}")
print(f"\nSample data:")
df.head()

## 2. Naive Comparison (BIASED)

In [ ]:
# Simple comparison
with_advisor = df[df['has_advisor'] == 1]['sales_lakhs'].mean()
without_advisor = df[df['has_advisor'] == 0]['sales_lakhs'].mean()
naive_effect = with_advisor - without_advisor

print("Naive Comparison (BIASED)")
print("=" * 50)
print(f"\nAverage sales WITH advisor: Rs {with_advisor:.1f} Lakhs")
print(f"Average sales WITHOUT advisor: Rs {without_advisor:.1f} Lakhs")
print(f"\nNaive effect estimate: Rs {naive_effect:.1f} Lakhs")
print(f"True effect: Rs 5.0 Lakhs")
print(f"\nBias: Rs {naive_effect - 5:.1f} Lakhs (OVERESTIMATE!)")

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(['Without Advisor', 'With Advisor'], [without_advisor, with_advisor], 
       color=['#3498db', '#2ecc71'])
ax.set_ylabel('Average Monthly Sales (Rs Lakhs)')
ax.set_title('Naive Comparison of Store Sales')
for i, v in enumerate([without_advisor, with_advisor]):
    ax.text(i, v + 0.5, f'Rs {v:.1f}L', ha='center', fontsize=12)
plt.show()

## 3. Check Baseline Imbalance

Let's see WHY the naive comparison is biased - the groups are different!

In [ ]:
def check_balance(df, treatment_col='has_advisor'):
    """Check covariate balance between treated and control groups."""
    covariates = ['footfall', 'store_size', 'is_mall', 'city_tier']
    
    results = []
    for cov in covariates:
        treated = df[df[treatment_col] == 1][cov]
        control = df[df[treatment_col] == 0][cov]
        
        pooled_std = np.sqrt((treated.std()**2 + control.std()**2) / 2)
        smd = abs(treated.mean() - control.mean()) / pooled_std if pooled_std > 0 else 0
        
        results.append({
            'Covariate': cov,
            'Treated Mean': treated.mean(),
            'Control Mean': control.mean(),
            'SMD': smd,
            'Balanced?': 'Yes' if smd < 0.1 else 'NO'
        })
    
    return pd.DataFrame(results)

balance_before = check_balance(df)

print("Covariate Balance BEFORE Matching")
print("=" * 70)
print(balance_before.to_string(index=False))
print("\nSMD > 0.1 indicates imbalance (selection bias)!")

In [ ]:
# Visualize imbalance
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

covariates = ['footfall', 'store_size', 'is_mall', 'city_tier']
titles = ['Daily Footfall', 'Store Size (sqft)', 'Mall Location', 'City Tier']

for ax, cov, title in zip(axes.flat, covariates, titles):
    treated = df[df['has_advisor'] == 1][cov]
    control = df[df['has_advisor'] == 0][cov]
    
    if cov in ['is_mall', 'city_tier']:
        # Bar chart for categorical
        x = np.arange(len(treated.unique()))
        width = 0.35
        ax.bar(x - width/2, treated.value_counts(normalize=True).sort_index(), 
               width, label='With Advisor', color='#2ecc71')
        ax.bar(x + width/2, control.value_counts(normalize=True).sort_index(), 
               width, label='Without Advisor', color='#3498db')
        ax.set_xticks(x)
        ax.set_ylabel('Proportion')
    else:
        # Histogram for continuous
        ax.hist(control, bins=20, alpha=0.5, label='Without Advisor', color='#3498db')
        ax.hist(treated, bins=20, alpha=0.5, label='With Advisor', color='#2ecc71')
        ax.set_ylabel('Count')
    
    ax.set_title(title)
    ax.legend()

plt.suptitle('Covariate Distributions: Treated vs Control (BEFORE Matching)', y=1.02)
plt.tight_layout()
plt.show()

## 4. Estimate Propensity Scores

Propensity score = P(Treatment | Covariates)

We model the probability of having a beauty advisor given store characteristics.

In [ ]:
# Fit propensity score model
X = df[['footfall', 'store_size', 'is_mall', 'city_tier']]
y = df['has_advisor']

ps_model = LogisticRegression(max_iter=1000)
ps_model.fit(X, y)

# Add propensity scores to dataframe
df['propensity_score'] = ps_model.predict_proba(X)[:, 1]

print("Propensity Score Distribution")
print("=" * 50)
print(f"\nTreated stores:")
print(f"  Mean PS: {df[df['has_advisor']==1]['propensity_score'].mean():.3f}")
print(f"  Range: [{df[df['has_advisor']==1]['propensity_score'].min():.3f}, {df[df['has_advisor']==1]['propensity_score'].max():.3f}]")

print(f"\nControl stores:")
print(f"  Mean PS: {df[df['has_advisor']==0]['propensity_score'].mean():.3f}")
print(f"  Range: [{df[df['has_advisor']==0]['propensity_score'].min():.3f}, {df[df['has_advisor']==0]['propensity_score'].max():.3f}]")

In [ ]:
# Visualize propensity score distributions
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(df[df['has_advisor']==0]['propensity_score'], bins=30, alpha=0.5, 
        label='Without Advisor', color='#3498db', density=True)
ax.hist(df[df['has_advisor']==1]['propensity_score'], bins=30, alpha=0.5, 
        label='With Advisor', color='#2ecc71', density=True)

ax.set_xlabel('Propensity Score')
ax.set_ylabel('Density')
ax.set_title('Propensity Score Distribution by Treatment Status')
ax.legend()
ax.axvline(x=0.2, color='red', linestyle='--', alpha=0.5)
ax.axvline(x=0.8, color='red', linestyle='--', alpha=0.5)
ax.text(0.5, ax.get_ylim()[1]*0.9, 'Common Support Region', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

## 5. Propensity Score Matching

For each treated store, find the most similar control store based on propensity score.

In [ ]:
def match_stores(df, caliper=0.1):
    """Perform 1:1 nearest neighbor matching."""
    treated = df[df['has_advisor'] == 1].copy()
    control = df[df['has_advisor'] == 0].copy()
    
    # Fit nearest neighbors on control group
    nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
    nn.fit(control[['propensity_score']])
    
    # Find matches
    distances, indices = nn.kneighbors(treated[['propensity_score']])
    
    # Apply caliper (max allowed distance)
    valid_matches = distances.flatten() <= caliper
    
    matched_treated = treated[valid_matches].copy()
    matched_control_idx = indices[valid_matches].flatten()
    matched_control = control.iloc[matched_control_idx].copy()
    
    return matched_treated, matched_control

matched_treated, matched_control = match_stores(df, caliper=0.1)

print("Matching Results")
print("=" * 50)
print(f"\nOriginal treated stores: {df['has_advisor'].sum()}")
print(f"Matched pairs: {len(matched_treated)}")
print(f"Unmatched (dropped): {df['has_advisor'].sum() - len(matched_treated)}")

## 6. Check Balance After Matching

In [ ]:
# Create matched dataframe
matched_df = pd.concat([
    matched_treated.assign(has_advisor=1),
    matched_control.assign(has_advisor=0)
])

balance_after = check_balance(matched_df)

print("Covariate Balance AFTER Matching")
print("=" * 70)
print(balance_after.to_string(index=False))
print("\nAll SMD values should now be < 0.1 (balanced!)")

In [ ]:
# Compare balance before vs after
fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(len(balance_before))
width = 0.35

ax.bar(x - width/2, balance_before['SMD'], width, label='Before Matching', color='#e74c3c')
ax.bar(x + width/2, balance_after['SMD'], width, label='After Matching', color='#2ecc71')

ax.axhline(y=0.1, color='black', linestyle='--', label='Balance Threshold (0.1)')
ax.set_xlabel('Covariate')
ax.set_ylabel('Standardized Mean Difference (SMD)')
ax.set_title('Covariate Balance: Before vs After Matching')
ax.set_xticks(x)
ax.set_xticklabels(balance_before['Covariate'])
ax.legend()

plt.tight_layout()
plt.show()

## 7. Estimate Causal Effect

In [ ]:
# Calculate ATE from matched sample
ate = matched_treated['sales_lakhs'].mean() - matched_control['sales_lakhs'].mean()

# Standard error
n = len(matched_treated)
se = np.sqrt(
    matched_treated['sales_lakhs'].var() / n +
    matched_control['sales_lakhs'].var() / n
)

# 95% CI
ci_lower = ate - 1.96 * se
ci_upper = ate + 1.96 * se

print("Causal Effect Estimate (After Matching)")
print("=" * 50)
print(f"\nMatched sales WITH advisor: Rs {matched_treated['sales_lakhs'].mean():.1f} Lakhs")
print(f"Matched sales WITHOUT advisor: Rs {matched_control['sales_lakhs'].mean():.1f} Lakhs")
print(f"\nEstimated causal effect: Rs {ate:.2f} Lakhs")
print(f"Standard error: Rs {se:.2f} Lakhs")
print(f"95% CI: [Rs {ci_lower:.2f}, Rs {ci_upper:.2f}] Lakhs")

In [ ]:
# Final comparison
print("\n" + "=" * 50)
print("COMPARISON: Naive vs Matched Estimates")
print("=" * 50)
print(f"\nTrue effect:           Rs 5.00 Lakhs")
print(f"Naive estimate:        Rs {naive_effect:.2f} Lakhs (bias: +{naive_effect - 5:.2f})")
print(f"Matched estimate:      Rs {ate:.2f} Lakhs (error: {ate - 5:+.2f})")
print(f"\nBias reduction: {abs(naive_effect - 5) - abs(ate - 5):.2f} Lakhs")
print(f"\nPropensity score matching significantly reduces selection bias!")

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(10, 6))

estimates = ['True Effect', 'Naive Estimate', 'PSM Estimate']
values = [5.0, naive_effect, ate]
colors = ['#2ecc71', '#e74c3c', '#3498db']

bars = ax.bar(estimates, values, color=colors)
ax.axhline(y=5.0, color='green', linestyle='--', alpha=0.5, label='True Effect')
ax.set_ylabel('Effect Size (Rs Lakhs)')
ax.set_title('Comparison of Effect Estimates')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.3, f'Rs {val:.1f}L', 
            ha='center', fontsize=11)

plt.tight_layout()
plt.show()

## Key Takeaways

1. **Selection bias** occurs when treatment is not randomly assigned
2. **Naive comparisons overestimate** effects when better units get treatment
3. **Propensity scores** = P(Treatment | Covariates)
4. **Matching on PS** creates comparable groups
5. **Check balance** - SMD < 0.1 after matching
6. **PSM reduces but doesn't eliminate bias** - unmeasured confounders remain
7. **Gold standard is still randomization** - PSM is for when experiments aren't possible